This section imports necessary tools and standard C++ libraries 

In [ ]:
#include <TTree.h>
#include <TFile.h>
#include <TDatabasePDG.h>
#include <TLorentzVector.h>
#include <TMath.h>
#include <TCanvas.h>
#include <TBenchmark.h>
#include <iostream>

Headers specific to ROOT and CLAS12 for reading .hipo files

In [ ]:
#include "clas12reader.h"
#include "HipoChain.h"

A shorcut that tells the compiler to automatically look inside the Standard Library (std) whenever it encounters a command it doesn't immediately recognize

In [ ]:
using namespace std;

A translation tool that takes raw data structure from CLAS12 software and converts it into a 4 vector for ROOT software to understand

p4.SetXYZM(...): This is a specific command belonging to the ROOT TLorentzVector class. It tells the vector: "I am going to give you four numbers: the X-momentum, Y-momentum, Z-momentum, and the invariant Mass. Use these to build the full 4-momentum vector

rp->par()->getPx(): This is how C++ extracts data from the CLAS12 pointer.

rp is the particle.

->par() accesses the base kinematics properties of that particle.

->getPx() fetches the specific momentum value along the X-axis in GeV/c

In [ ]:
void SetLorentzVector(TLorentzVector &p4, clas12::region_part_ptr rp)
{
    p4.SetXYZM(rp->par()->getPx(), rp->par()->getPy(), rp->par()->getPz(), p4.M());
}

void: This function does not return a new variable. It just modifies something that already exists

A correction factor is applied to account for detector efficiencies     

In [ ]:
TLorentzVector CorrectElectron(TLorentzVector &p4)

defines a function named CorrectElectron to measure momentum and energy of an electron

In [ ]:
Double_t E_cor, px_el, py_el, pz_el;
TLorentzVector el_new;

E_cor = p4.E()
      + 0.085643
      - 0.0288063 * p4.E()
      + 0.00894691 * p4.E() * p4.E()
      - 0.000725449 * p4.E() * p4.E() * p4.E();

E_cor = corrected energy
p4.E() = gets raw uncorrected energy of electron directly from 4 vector

this is a 3rd degree polynomial that represents the curve fit from plotting the difference between the true energy and measured energy

In [ ]:
px_el = E_cor * (p4.Px() / p4.Rho());
py_el = E_cor * (p4.Py() / p4.Rho());
pz_el = E_cor * (p4.Pz() / p4.Rho());

used to change the x,y,z momentum vectors
p4.Rho() calculates the magnitude of the 3-momentum

stretching the momentum vectors out to their new, correct lengths without changing the direction the electron was flying.

In [ ]:
el_new.SetXYZM(px_el, py_el, pz_el, 0.000511);
return el_new;

Building the new vector el.new

In [ ]:
struct ParticleInfo {
    int   pid;
    int   charge;
    float px;
    float py;
    float pz;
    float P_mag;
    float vx;
    float vy;
    float vz;
    float theta;
    float phi;
    float deltaTime;
    float beta;
    float betafromP;
    float path;
    int   region;
    int   status;
    float chi2pid;
};

struct = custom storage container or a blueprint for a data profile

Instead of passing around 18 different loose variables for every single particle, this struct groups them all together into one neat, organized package called "ParticleInfo"

Now to define a new function called "getParticle" to extract data from .hipo file and pack it into "ParticleInfo"

In [ ]:
void getParticle(ParticleInfo &info, const clas12::region_part_ptr particle)
{
    info.pid        = particle->getPid();
    info.P_mag      = particle->getP();
    info.px         = particle->getPx();
    info.py         = particle->getPy();
    info.pz         = particle->getPz();
    info.vx         = particle->par()->getVx();
    info.vy         = particle->par()->getVy();
    info.vz         = particle->par()->getVz();
    info.theta      = particle->getTheta();
    info.phi        = particle->getPhi();
    info.deltaTime  = particle->getDeltaTime();
    info.beta       = particle->getBeta();
    info.betafromP  = particle->getBetaFromP();
    info.region     = particle->getRegion();
    info.status     = particle->getStatus();
    info.chi2pid    = particle->getChi2Pid();
    info.charge     = particle->par()->getCharge();  // <-- added line to get charge
}

 the function is taking the specific, real-world container and physically putting data inside it without accidentally changing or deleting it

In [ ]:
void writeParticleInfoToTree(ParticleInfo &info, TTree *tree, const std::string &suffix)
{
    tree->Branch(("pid_" + suffix).c_str(), &info.pid, ("pid_" + suffix + "/I").c_str());
    tree->Branch(("charge_" + suffix).c_str(), &info.charge, ("charge_" + suffix + "/I").c_str());
    tree->Branch(("px_" + suffix).c_str(), &info.px, ("px_" + suffix + "/F").c_str());
    tree->Branch(("py_" + suffix).c_str(), &info.py, ("py_" + suffix + "/F").c_str());
    tree->Branch(("pz_" + suffix).c_str(), &info.pz, ("pz_" + suffix + "/F").c_str());
    tree->Branch(("P_mag_" + suffix).c_str(), &info.P_mag, ("P_mag_" + suffix + "/F").c_str());
    tree->Branch(("vx_" + suffix).c_str(), &info.vx, ("vx_" + suffix + "/F").c_str());
    tree->Branch(("vy_" + suffix).c_str(), &info.vy, ("vy_" + suffix + "/F").c_str());
    tree->Branch(("vz_" + suffix).c_str(), &info.vz, ("vz_" + suffix + "/F").c_str());
    tree->Branch(("theta_" + suffix).c_str(), &info.theta, ("theta_" + suffix + "/F").c_str());
    tree->Branch(("phi_" + suffix).c_str(), &info.phi, ("phi_" + suffix + "/F").c_str());
    tree->Branch(("deltaTime_" + suffix).c_str(), &info.deltaTime, ("deltaTime_" + suffix + "/F").c_str());
    tree->Branch(("beta_" + suffix).c_str(), &info.beta, ("beta_" + suffix + "/F").c_str());
    tree->Branch(("betafromP_" + suffix).c_str(), &info.betafromP, ("betafromP_" + suffix + "/F").c_str());
    tree->Branch(("region_" + suffix).c_str(), &info.region, ("region_" + suffix + "/I").c_str());
    tree->Branch(("status_" + suffix).c_str(), &info.status, ("status_" + suffix + "/I").c_str());
    tree->Branch(("chi2pid_" + suffix).c_str(), &info.chi2pid, ("chi2pid_" + suffix + "/F").c_str());
}   

Third and final prep function to take data that was just packed into the "ParticleInfo" struct and plugged into a ROOT TTree

TTree is ROOT's version of spreadsheet or database

By passing a "suffix" (like "e", "p1", "p2", or "pim"), this function uses string concatenation ("px_" + suffix) to dynamically generate column names

Now for main function and looping;

In [ ]:
void v2_hipo_root_pppim()

When you run a script in ROOT, the main function must have the exact same name as the file itself

In [ ]:
    auto db = TDatabasePDG::Instance();

Create connection to ROOT's built-in encyclopedia to particle physics

In [ ]:
    // --- Event counters ---
    Long64_t n_total        = 0;  // all events in the HIPO file
    Long64_t n_have_topo    = 0;  // events with 1e, 2p, 1π-
    Long64_t n_status_ok    = 0;  // plus electron status < 0
    Long64_t n_filled_tree  = 0;  // events that actually go to the TTree
    Long64_t n_has_neutral  = 0;  // events where has_neutral == 1

Long64_t is a special variable type provided by ROOT. It stands for a "64-bit integer," meaning it can count up to 9.2 quintillion

these counters act like a funnel, tracking how many events survive each round of cuts. At the very end these numbers are printed to the terminal so you know exactly what percentage of the raw data was actually useful

n_status_ok: Once you find an event with the right particles, you check the electron's detector "status" code

n_has_neutral: After calculating all the missing momentum, your script looks at the calorimeter to see if there is leftover energy that could be a neutral particle (like a photon or neutron)

- [The only way the detector knows a neutral particle exists is when it finally crashes into the outer Electromagnetic Calorimeter (ECAL) and leaves a splash of energy]

- When the code successfully finds one of these matching mystery splashes, it sets the flag has_neutral = 1

n_filled_tree: This is the final step. If an event survives all the previous checks, the data is officially written (filled) into your ROOT TTree

In [ ]:
    Double_t mass_e   = db->GetParticle(11)->Mass();
    Double_t mass_p   = db->GetParticle(2212)->Mass();
    Double_t mass_pim = db->GetParticle(211)->Mass();

Instead of manually typing mass_e = 0.000511 (and potentially making a typo or using a slightly outdated measurement), the code uses the db variable to look up particle #11 (the electron) in the database and extract its exact mass

In [ ]:
Double_t energy = 10.1998;

defines the beam energy used in your physics experiment, it is the exact kinetic energy of the electron beam that was fired into the target during your specific dataset's run (Spring 2019 at Jefferson Lab's CLAS12 detector

In [ ]:
    TLorentzVector beam  (0, 0, sqrt(energy * energy - mass_e * mass_e), energy);

This line constructs the initial state of the incoming electron beam. It assumes the beam is traveling perfectly straight down the Z-axis (hence 0 for X and 0 for Y momentum). It then calculates the exact Z-momentum using the relativistic energy-momentum relation

TLorentzVector is an object in ROOT that holds four numbers: the X, Y, and Z momentum, and the Energy (or Mass).

In [ ]:
    TLorentzVector target(0, 0, 0, db->GetParticle(2212)->Mass());
    TLorentzVector p_electron(0, 0, 0, db->GetParticle(11)->Mass());
    TLorentzVector p_proton1 (0, 0, 0, db->GetParticle(2212)->Mass());
    TLorentzVector p_proton2 (0, 0, 0, db->GetParticle(2212)->Mass());
    TLorentzVector p_pim     (0, 0, 0, db->GetParticle(211)->Mass());

This block of code is creating the initial "empty" 4-vectors for the particles you expect to find in every collision event.

In [ ]:
    clas12root::HipoChain chain;
    auto  config_c12 = chain.GetC12Reader();
    auto &c12        = chain.C12ref();

    chain.Add("/lustre24/expphy/volatile/clas12/leomart/Data/Runs/Spring2019/FT_merged/Pp_eFT_all.hipo");

"chainning" the all hipo files together with [HipoChain]

In [ ]:
    Double_t pp_inv_mass, miss_mass, miss_mass_sq;
    TLorentzVector p_electron_cor;

These two lines simply declare a few empty variables that will be used heavily during the main physics loop

In [ ]:
    float Ecal_e  = 0, Pcal_e  = 0;
    float Ecal_p1 = 0, Ecal_p2 = 0;
    float Ecal_pim = 0;

    float Enbar_calo  = 0.0f;
	float neutral_angle = -999.0f;
    int   has_neutral = 0;

	float e_status_val = 0;

This block of code is setting up the storage variables for the Electromagnetic Calorimeter (ECAL) data and detector status flags.

- Pcal stands for the "Pre-shower Calorimeter" (the very first, thin layer).
- Ecal stands for the main Electromagnetic Calorimeter (the deeper, thicker layers)
- Enbar_calo: Stores the raw energy of the mystery hit in the calorimeter ("Enbar" often stands for Energy of a neutral particle/baryon).
- has_neutral: A simple binary switch. 0 means "no neutral particle found," and 1 means "yes, we found one!"
- If a particle doesn't exist, its angle isn't 0 degrees (0 degrees means it is flying perfectly straight down the beamline). By setting it to an impossibly negative number like -999, when you plot your data later, any "failed" or "empty" events will clump up way off the side of the graph at -999, making it super easy to visually filter out the bad data!

In [ ]:
float e_status_val = 0;

the status code tells you exactly which sub-systems of the detector the electron passed through, for example, a negative status code < 0 means it was recorded in the Forward Detector

In [ ]:
TFile *file       = new TFile("v2_kev_Pppim_eFT_all.root", "RECREATE");
TTree *tree_indiv = new TTree("Individual", "Individual particle variables");

These two lines set up the output file where all your physics analysis results will be saved.

- TFile: This is a ROOT class used to manage files on your hard drive
- "RECREATE": This is a crucial safety instruction. It tells the computer: "Create a new file with this name. If a file with this name already exists in this folder, overwrite it and start fresh". (If you used "UPDATE" instead, it would add new data to the bottom of an existing file).

- "Individual": This is the official "internal name" of the tree. When you open this .root file later to plot your histograms, you will tell ROOT, "Go inside the file and find the tree named 'Individual'."

- the tree_indiv->Fill() command inside the main loop, it is writing the data directly into this TFile


A TTree doesn't actually store data directly inside itself; instead, it stores data inside individual objects called TBranches. When you want to save a specific variable to your file, you have to create a branch for it and formally link that branch to the variable in your computer's memory.

The & symbol means you are giving ROOT the memory address of the miss_mass variable you created earlier in your script.

When your main while loop runs, it constantly overwrites the miss_mass variable with new calculations for every single collision event. At the very end of the loop, you will call the command tree_indiv->Fill(). Because you set up these branches, the Fill() command knows exactly where to look

In [ ]:
    tree_indiv->Branch("miss_mass",     &miss_mass);
    tree_indiv->Branch("miss_mass_sq",  &miss_mass_sq);
    tree_indiv->Branch("pp_inv_mass",   &pp_inv_mass);

    ParticleInfo electronInfo, proton1Info, proton2Info, piminusInfo;

    writeParticleInfoToTree(electronInfo, tree_indiv, "e");
    writeParticleInfoToTree(proton1Info,  tree_indiv, "p1");
    writeParticleInfoToTree(proton2Info,  tree_indiv, "p2");
    writeParticleInfoToTree(piminusInfo,  tree_indiv, "pim");

In [ ]:
    tree_indiv->Branch("Ecal_e",   &Ecal_e,   "Ecal_e/F");
    tree_indiv->Branch("Pcal_e",   &Pcal_e,   "Pcal_e/F");
    tree_indiv->Branch("Ecal_p1",  &Ecal_p1,  "Ecal_p1/F");
    tree_indiv->Branch("Ecal_p2",  &Ecal_p2,  "Ecal_p2/F");
    tree_indiv->Branch("Ecal_pim", &Ecal_pim, "Ecal_pim/F");

	tree_indiv->Branch("e_status", &e_status_val, "e_status/F");

    tree_indiv->Branch("Enbar_calo",  &Enbar_calo,  "Enbar_calo/F");
	tree_indiv->Branch("neutral_angle", &neutral_angle, "neutral_angle/F");
    tree_indiv->Branch("has_neutral", &has_neutral, "has_neutral/I");

Manual way of creating branches by allocating space on drive. 
- /F means 32-bit float
- /I mean 32-bit integer

In [ ]:
    while (chain.Next()) {

chain.Next() is a built-in command belonging to the HipoChain tool from the clas12root library;

The reader moves forward exactly one "event" (one recorded proton collision) inside the .hipo data file

It takes all the data from that collision (the particles found, their momenta, their calorimeter hits) and permanently loads it into that live c12 reference variable you created earlier.

In [ ]:
        p_electron.SetXYZM(0, 0, 0, mass_e);
        p_proton1.SetXYZM (0, 0, 0, mass_p);
        p_proton2.SetXYZM (0, 0, 0, mass_p);
        p_pim.SetXYZM     (0, 0, 0, mass_pim);

resetting 4-vectors

In [ ]:
        auto electrons = c12->getByID(11);
        auto protons   = c12->getByID(2212);
        auto piminus   = c12->getByID(-211);

They extract the specific particles you want out of the massive pile of collision data
- c12->getByID(...) is a search function built into the CLAS12 software
- By using auto (which tells C++ to automatically format the variable), the variables electrons, protons, and piminus become arrays (or vectors) of pointers.

In [ ]:
        if (electrons.size() < 1 || protons.size() < 2 || piminus.size() < 1) continue;

Cutt for right topology (e, p, p, pim)

In [ ]:
        bool is_valid_electron = (std::abs(e_status) >= 1000 && std::abs(e_status) < 4000);

This line is a safety check to ensure that the electron your code found is actually a real, reliable particle, and not just background noise or a bad measurement

In the CLAS12 detector software, every particle reconstructed by the computer is assigned a status integer. This number isn't just a random label; it acts like a barcode that tells you exactly which physical parts of the massive detector the particle passed through.

- Status < 0 or 1000 - 4000: Usually indicates the particle was recorded in the Forward Detector (FD 1000 - 1999) or Forward Tagger (FT 2000 - 3999), which are the highly precise sections of the detector designed specifically to catch fast-moving, forward-scattering electrons

- Status 4000+: Usually indicates the particle was recorded in the Central Detector (CD), which surrounds the target at wider angles and is generally used for slower particles like protons or pions.

The line: (std::abs(e_status) >= 1000 && std::abs(e_status) < 4000) translates to:
"Look at the electron's status code. Disregard whether it is positive or negative. Just check if the core number is between 1000 and 3999."

By forcing the electron's status to be in the 1000-3999 range, you guarantee that the electron you are using for your Missing Mass calculation hit the high-quality Forward detectors

https://code.jlab.org/hallb/clas12/coatjava/coatjava/-/blob/10.0.2/reconstruction/eb/doc/dst.md

The exact definitions for these status codes come directly from the CLAS12 Data Summary Tape (DST) structure, which is standardized by Jefferson Lab's official reconstruction software (called coatjava).



In [ ]:
                SetLorentzVector(p_electron, electrons[0]);
                SetLorentzVector(p_proton1,  protons[0]);
                SetLorentzVector(p_proton2,  protons[1]);
                SetLorentzVector(p_pim,      piminus[0]);

This is just calling the function and filling the data columns

Once these four lines of code finish executing, you now have four fully populated TLorentzVector objects. You know exactly how fast they are moving, what direction they are flying, and what their total energies are.

In [ ]:
				e_status_val = (float)electrons[0]->getStatus();

By explicitly writing (float) right here in the loop, you guarantee that the data matches the exact format the database is expecting!

In [ ]:
				// --- 3. THE MOMENTUM CORRECTION CHANGE ---
                TLorentzVector p_electron_final;
                if (e_status < 0) {
                    // It's in the Forward Detector, apply your correction
                    p_electron_final = CorrectElectron(p_electron);
                } else {
                    // It's in the Forward Tagger, use raw momentum
                    p_electron_final = p_electron;
                }

A correction does not throw any data away. Instead, it takes data that is already considered "good" and mathematically tweaks it to fix known hardware inaccuracies (like a misaligned magnetic field or energy lost in the detector materials).

This code is making a decision on how to apply the mathematical correction based on where the electron was physically located in the detector.

The Forward Tagger is built differently than the Forward Detector and doesn't suffer from the same energy loss issues. Therefore, you don't need to apply the polynomial correction

In [ ]:
                TLorentzVector MM = beam + target - p_electron_final - p_proton1 - p_proton2 - p_pim;
                miss_mass    = MM.M();
                miss_mass_sq = MM.M2();
                pp_inv_mass  = (p_proton1 + p_proton2).M();

This is building the missing mass data columns!

The Missing Mass (miss_mass) tells you what you didn't see. The Invariant Mass (pp_inv_mass) tells you exactly what you did see. Together, they allow a physicist to map out the entire collision

Physicists look at both because glancing blows (FT) and hard hits (FD) probe entirely different aspects of the Strong Nuclear Force

In [ ]:
// %%%%%%%%%%%%%%%%%%%%%%%% Calorimeter Block %%%%%%%%%%%%%%%%%%%%%%%%%%%%
                auto &calos = c12->getRECCalorimeter();

                int ie  = electrons[0]->getIndex();
                int ip1 = protons[0]->getIndex();
                int ip2 = protons[1]->getIndex();
                int ipi = piminus[0]->getIndex();

                // Ecal_e  = Pcal_e  = 0.0f;
                // Ecal_p1 = Ecal_p2 = 0.0f;
                // Ecal_pim = 0.0f;
                // Enbar_calo  = 0.0f;
                // has_neutral = 0;

				Enbar_calo = 0.0f;
				neutral_angle = -999.0f;
				has_neutral = 0;

                TVector3 p_miss = MM.Vect();
                double   best_angle = 1e9;

                for (int i = 0; i < calos.getRows(); i++) {
                    calos.setEntry(i);

                    int   detector = calos.getDetector();  // 7 for ECAL system
                    int   pindex   = calos.getPindex();
                    int   layer    = calos.getLayer();  // 1 for PCAL, 4-6 for INNER/OUTER ECAL
                    float Ehit     = calos.getEnergy();
                    float x        = calos.getX();
                    float y        = calos.getY();
                    float z        = calos.getZ();			
                    
					if (detector != 7) continue;

                    if (pindex == ie) {
                        if (layer == 1)      Pcal_e  += Ehit;
                        else if (layer >= 4) Ecal_e  += Ehit;
                    }
                    else if (pindex == ip1 && layer >= 4) Ecal_p1 += Ehit;
                    else if (pindex == ip2 && layer >= 4) Ecal_p2 += Ehit;
                    else if (pindex == ipi && layer >= 4) Ecal_pim += Ehit;

					// Identify if this belongs to our primary 4 tracks
    				bool is_primary = (pindex == ie || pindex == ip1 || pindex == ip2 || pindex == ipi);


                    if (!is_primary) {
                        TVector3 r_hit(x, y, z);
                        double angle = r_hit.Angle(p_miss);

                        if (angle < best_angle) {
                            best_angle  = angle;
                            Enbar_calo  = Ehit;
							neutral_angle = (float)angle;
                            has_neutral = 1;
                        }
                    }
                }
                // %%%%%%%%%%%%%%%%%%%%%%%% Calorimeter Block:End %%%%%%%%%%%%%%%%%%%%%%%%%


First access the calorimeter; 
- auto &calos = c12->getRECCalorimeter();

Then get Particle Index;
- int ie  = electrons[0]->getIndex();

To understand which energy splash belongs to which particle, you use getIndex().
Every particle the software finds is given a master ID number (an index) for that specific collision event. If the electron is particle #1, the proton is particle #2, etc., you save those index numbers into variables (ie, ip1, etc.) so you can match them up with the calorimeter hits

Get detector, Pindex, layer, Ehit, x, y, z info

If the hit isn't in Detector 7 then the for loop skips it
- 					if (detector != 7) continue;

Add up the energy of the known particles (e, p, p, pim)
 - if (pindex -- ie) ...

 - else if(...)


Finding the neutral particle, If the hit matched any of your four known particles, is_primary becomes true.
For false hits, the software couldn't assign this hit to a particle (untracked hits)

Taking the x,y,z of the untracked hit and turn it into a 3D vector (r_hit) pointing from the target to the detector wall. Then calculate the angle between the untracked hits and the missing mass vector that was calculated earlier (p_miss)

If this angle is incredibly small (meaning the physical splash happened exactly where the missing momentum was pointing), it overwrites best_angle. It saves the energy of the hit (Enbar_calo), saves the angle (neutral_angle), and officially flips the has_neutral switch to 1

"Which one of these splashes lines up best with the direction the antineutron was supposed to be flying?"
Kinematic Matching

Jsut added "orphaned hits" to collect all untracked hits with my topology

The rest is just filling the data containers; getParticle(...) 

Saving the row to the database; n_filled_tree++

safely saving the file; tree_indiv->Write();

In [ ]:
    std::cout << "Total events in HIPO:          " << n_total       << std::endl;
    std::cout << "Events with 1e2p1pi- topology: " << n_have_topo   << std::endl;
    std::cout << "Events with status OK:         " << n_status_ok   << std::endl;
    std::cout << "Events written to TTree:       " << n_filled_tree << std::endl;
    std::cout << "Events with has_neutral == 1:  " << n_has_neutral << std::endl;

these are just to print the numbers of interest in the terminal after the macro is finished